# Supervised Classification & Tuning - Sanika Hajare
- 4 Models: LogisticRegression, RandomForest, XGBoost, LightGBM
- GridSearchCV + Stratified K-Fold + ROC-AUC +.pkl

In [ ]:
import pandas as pd, numpy as np, joblib, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay, precision_recall_fscore_support
np.random.seed(42)

# Synthetic Data (replace with your preprocessed data)
df=pd.DataFrame({'Age':np.random.randint(21,70,1000),'Income':np.random.normal(50000,15000,1000),'CreditScore':np.random.normal(650,100,1000),'LoanAmount':np.random.normal(15000,5000,1000),'EmploymentYears':np.random.randint(0,30,1000),'Gender':np.random.choice(['Male','Female'],1000),'Education':np.random.choice(['Graduate','Undergrad','PhD'],1000),'Married':np.random.choice(['Yes','No'],1000),'Default':np.random.choice([0,1],1000,p=[0.8,0.2])})
for c in ['Income','CreditScore','Education']: df.loc[df.sample(frac=0.05).index,c]=np.nan
X=df.drop('Default',1); y=df['Default']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
num=['Age','Income','CreditScore','LoanAmount','EmploymentYears']; cat=['Gender','Education','Married']
pre=ColumnTransformer([('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())]),num),('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])

models={
'LogisticRegression': (LogisticRegression(max_iter=1000), {'classifier__C':[0.1,1,10]}),
'RandomForest': (RandomForestClassifier(random_state=42), {'classifier__n_estimators':[100,200], 'classifier__max_depth':[5,10,None]}),
'XGBoost': (XGBClassifier(eval_metric='logloss',random_state=42), {'classifier__n_estimators':[100,200], 'classifier__max_depth':[3,6], 'classifier__learning_rate':[0.05,0.1]}),
'LightGBM': (LGBMClassifier(random_state=42), {'classifier__n_estimators':[100,200], 'classifier__learning_rate':[0.05,0.1]})
}
results=[]; best_estimators={}
skf=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name,(clf,params) in models.items():
 pipe=Pipeline([('preprocessor',pre),('classifier',clf)])
 gs=GridSearchCV(pipe, params, cv=skf, scoring='roc_auc', n_jobs=-1, verbose=0)
 gs.fit(X_train,y_train)
 y_pred=gs.predict(X_test); y_proba=gs.predict_proba(X_test)[:,1]
 prec,rec,f1,_=precision_recall_fscore_support(y_test,y_pred,average='binary')
 auc=roc_auc_score(y_test,y_proba)
 results.append([name,gs.best_params_,prec,rec,f1,auc])
 best_estimators[name]=gs.best_estimator_
 print(f"{name} | P:{prec:.3f} R:{rec:.3f} F1:{f1:.3f} AUC:{auc:.3f} | Best:{gs.best_params_}")
 print(classification_report(y_test,y_pred))

res_df=pd.DataFrame(results, columns=['Model','BestParams','Precision','Recall','F1','ROC_AUC']).sort_values('ROC_AUC', ascending=False)
res_df


In [ ]:
# Comparison Table + ROC Curves
fig, ax=plt.subplots(figsize=(8,6))
for name,est in best_estimators.items():
 RocCurveDisplay.from_estimator(est, X_test, y_test, name=name, ax=ax)
plt.title('ROC-AUC Comparison - 4 Models'); plt.show()
res_df


In [ ]:
# Confusion Matrices
fig, axes=plt.subplots(2,2, figsize=(10,8))
axes=axes.ravel()
for i,(name,est) in enumerate(best_estimators.items()):
 cm=confusion_matrix(y_test, est.predict(X_test))
 sns.heatmap(cm, annot=True, fmt='d', ax=axes[i]); axes[i].set_title(name)
plt.tight_layout(); plt.show()


In [ ]:
# Champion Model Selection & Save.pkl
champion_name=res_df.iloc[0]['Model']
champion=best_estimators[champion_name]
print(f"Champion Model: {champion_name} with AUC {res_df.iloc[0]['ROC_AUC']:.4f}")
joblib.dump(champion, f'{champion_name}_champion.pkl')
joblib.dump(res_df, 'model_comparison_table.pkl')
print(f"Saved: {champion_name}_champion.pkl")
